# Quality Control for the processed parquet

In [4]:
import pandas as pd
from pathlib import Path

root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

panel = pd.read_parquet(root / "data/processed_data/analysis_panel.parquet")
print(panel.shape)
print(panel.dtypes)
panel.head(20)

(17150, 58)
YEAR                               int16
GEOGRAPHY_CODE                  category
GEOGRAPHY_NAME                  category
IS8_SECTOR                      category
EMPLOYEES                        float32
BUSINESSES                       float32
gva_per_hour                     float64
weekly_pay                       float64
employment_rate                  float64
unemployment_rate                float64
gdhi_per_head                    float64
new_enterprises                  float64
deaths_of_enterprises            float64
active_enterprises               float64
high_growth_enterprises          float64
public_transport_to_employer     float64
drive_to_employer                float64
cycle_to_employer                float64
broadband_availability           float64
4g_area_coverage                 float64
ks2_attainment                   float64
gcse_by_age_19                   float64
ofsted                           float64
persistent_absences              float64
pers

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,EMPLOYEES,BUSINESSES,gva_per_hour,weekly_pay,employment_rate,unemployment_rate,...,emp_share,lq_emp,growth_emp,cagr_emp,growth_bus,cagr_bus,business_density,related_variety,size_large_share,size_micro_share
0,2016,E06000001,Hartlepool,Advanced Manufacturing,1470.0,25.0,29.84,521.2,69.7,4.6,...,0.048197,1.66415,0.010204,0.001693,-0.2,-0.036508,NaN,NaN,NaN,NaN
1,2016,E06000001,Hartlepool,Creative Industries,450.0,110.0,29.84,521.2,69.7,4.6,...,0.014754,0.298377,0.744444,0.097176,-0.090909,-0.01576,NaN,NaN,NaN,NaN
2,2016,E06000001,Hartlepool,Defence,0.0,0.0,29.84,521.2,69.7,4.6,...,0.0,0.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN
3,2016,E06000001,Hartlepool,Digital and Technologies,1510.0,440.0,29.84,521.2,69.7,4.6,...,0.049508,0.701919,-0.076159,-0.013116,-0.352273,-0.069823,NaN,NaN,NaN,NaN
4,2016,E06000001,Hartlepool,Financial Services,285.0,40.0,29.84,521.2,69.7,4.6,...,0.009344,0.157556,-0.070175,-0.012053,0.375,0.054509,NaN,NaN,NaN,NaN
5,2016,E06000001,Hartlepool,Life Sciences,40.0,0.0,29.84,521.2,69.7,4.6,...,0.001311,0.456549,0.0,0.0,<NA>,<NA>,NaN,NaN,NaN,NaN
6,2016,E06000001,Hartlepool,Professional and Business Services,2630.0,870.0,29.84,521.2,69.7,4.6,...,0.08623,0.483872,-0.019011,-0.003194,-0.264368,-0.049884,NaN,NaN,NaN,NaN
7,2016,E06000002,Middlesbrough,Advanced Manufacturing,620.0,25.0,29.50,481.9,68.8,5.1,...,0.010622,0.366756,0.153226,0.024045,0.4,0.057681,NaN,NaN,NaN,NaN
8,2016,E06000002,Middlesbrough,Creative Industries,1320.0,180.0,29.50,481.9,68.8,5.1,...,0.022614,0.457337,0.431818,0.06165,0.055556,0.009052,NaN,NaN,NaN,NaN
9,2016,E06000002,Middlesbrough,Defence,0.0,0.0,29.50,481.9,68.8,5.1,...,0.0,0.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN


In [5]:
# shape and coverage
print("Shape:", panel.shape)
print("\nYears:", sorted(panel["YEAR"].unique()))
print("LADs:", panel["GEOGRAPHY_CODE"].nunique())
print("Sectors:", sorted(panel["IS8_SECTOR"].unique()))

# missing values
missing = panel.isnull().sum()
missing_pct = (missing / len(panel) * 100).round(1)
print("\nMissing values:")
print(pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0"))

# indicator columns sanity check
print("\nKey indicator ranges:")
for col in ["lq_emp", "lq_bus", "growth_emp", "growth_bus", "cagr_emp", "cagr_bus"]:
    print(f"  {col}: min={panel[col].min():.3f}, max={panel[col].max():.3f}, nulls={panel[col].isnull().sum()}")

Shape: (17150, 58)

Years: [np.int16(2016), np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022)]
LADs: 350
Sectors: ['Advanced Manufacturing', 'Creative Industries', 'Defence', 'Digital and Technologies', 'Financial Services', 'Life Sciences', 'Professional and Business Services']

Missing values:
                              missing    pct
gva_per_hour                      196    1.1
weekly_pay                         98    0.6
employment_rate                   294    1.7
unemployment_rate                1029    6.0
gdhi_per_head                     196    1.1
new_enterprises                   196    1.1
deaths_of_enterprises             196    1.1
active_enterprises                196    1.1
high_growth_enterprises           196    1.1
public_transport_to_employer     3479   20.3
drive_to_employer                3479   20.3
cycle_to_employer                3479   20.3
broadband_availability            196    1.1
4g_area_coverage            

In [6]:
# check if working_age_pop column exists
print("working_age_pop" in panel.columns)
print([c for c in panel.columns if "work" in c.lower() or "pop" in c.lower() or "age" in c.lower()])

False
['4g_area_coverage', 'gcse_by_age_19']


In [8]:
print(f"LADs in panel: {panel['GEOGRAPHY_CODE'].nunique()}")

# load boundaries to compare
import geopandas as gpd
boundaries = gpd.read_file(root / "data/raw_data/boundaries/UK_Local_Authority_Districts_December_2023_Boundaries_UK_BGC_2537431731774104276.geojson")
print(f"LADs in boundaries: {boundaries['LAD23CD'].nunique()}")

# find missing LADs
missing_lads = set(boundaries['LAD23CD']) - set(panel['GEOGRAPHY_CODE'])
print(f"\nMissing LADs ({len(missing_lads)}):")
print(boundaries[boundaries['LAD23CD'].isin(missing_lads)][['LAD23CD', 'LAD23NM']])

LADs in panel: 350
LADs in boundaries: 361

Missing LADs (11):
       LAD23CD                               LAD23NM
296  N09000001               Antrim and Newtownabbey
297  N09000002  Armagh City, Banbridge and Craigavon
298  N09000003                               Belfast
299  N09000004              Causeway Coast and Glens
300  N09000005               Derry City and Strabane
301  N09000006                   Fermanagh and Omagh
302  N09000007               Lisburn and Castlereagh
303  N09000008                   Mid and East Antrim
304  N09000009                            Mid Ulster
305  N09000010                Newry, Mourne and Down
306  N09000011                   Ards and North Down


In [10]:
# confirm inf comes from zero start values
import numpy as np

emp_base_zeros = panel[panel["YEAR"] == 2016]["EMPLOYEES"].eq(0).sum()
bus_base_zeros = panel[panel["YEAR"] == 2016]["BUSINESSES"].eq(0).sum()
print(f"Zero EMPLOYEES in 2016: {emp_base_zeros}")
print(f"Zero BUSINESSES in 2016: {bus_base_zeros}")

# how many inf values
print(f"\ninf in growth_emp: {np.isinf(panel['growth_emp']).sum()}")
print(f"inf in growth_bus: {np.isinf(panel['growth_bus']).sum()}")

Zero EMPLOYEES in 2016: 322
Zero BUSINESSES in 2016: 455

inf in growth_emp: 203
inf in growth_bus: 238


In [3]:
# --- Missing values ---
print("=== MISSING VALUES (indicators panel) ===")
print(panel_ind.isnull().sum().to_string())

=== MISSING VALUES (indicators panel) ===
YEAR                                0
GEOGRAPHY_CODE                      0
GEOGRAPHY_NAME                      0
IS8_SECTOR                          0
EMPLOYEES                        2450
BUSINESSES                       2450
gva_per_hour                      308
weekly_pay                        154
employment_rate                   462
unemployment_rate                1617
gdhi_per_head                     308
new_enterprises                   308
deaths_of_enterprises             308
active_enterprises                308
high_growth_enterprises           308
uk_exports                      26950
inward_fdi                      26950
outward_fdi                     26950
goverment_randd                 26950
public_transport_to_employer     5467
drive_to_employer                5467
cycle_to_employer                5467
broadband_availability            308
4g_area_coverage                  308
ks2_attainment                  16786
gcse_by_

In [4]:
print(panel_ind['lq_emp'].describe())
print(f"\nSample lq_emp values:")
print(panel_ind[panel_ind['GEOGRAPHY_NAME'] == 'Sheffield']['lq_emp'].head(5))

count       24500.0
mean       0.931461
std        3.160158
min             0.0
25%        0.259797
50%        0.574108
75%        0.989644
max      120.030762
Name: lq_emp, dtype: Float64

Sample lq_emp values:
1715     0.47333
1716    0.661367
1717         0.0
1718    0.777725
1719    1.180456
Name: lq_emp, dtype: Float64


In [5]:
import sys
sys.path.insert(0, "..")

from src.indicators import IndicatorBuilder

builder = IndicatorBuilder(config)

# Test LQ on raw panel (with Total rows)
test = builder.compute_location_quotient(panel.copy())
print(f"Rows after LQ: {test.shape[0]:,}")
print(f"lq_emp null count: {test['lq_emp'].isnull().sum():,}")
print(f"lq_emp sample (Sheffield, Adv Mfg, 2023):")
print(test[
    (test['GEOGRAPHY_NAME'] == 'Sheffield') &
    (test['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (test['YEAR'] == 2023)
][['lq_emp']].values)

Rows after LQ: 26,950
lq_emp null count: 2,450
lq_emp sample (Sheffield, Adv Mfg, 2023):
[[0.49832895]]


In [6]:
test = panel.copy()
test = builder.compute_employment_share(test)
test = builder.compute_location_quotient(test)

print("Non-null lq_emp count:", test['lq_emp'].notna().sum())
print("Null lq_emp count:", test['lq_emp'].isna().sum())
print("\nlq_emp for Sheffield, Adv Mfg, 2023:")
print(test[
    (test['GEOGRAPHY_NAME'] == 'Sheffield') &
    (test['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (test['YEAR'] == 2023)
][['IS8_SECTOR', 'lq_emp']])

Non-null lq_emp count: 24500
Null lq_emp count: 2450

lq_emp for Sheffield, Adv Mfg, 2023:
                   IS8_SECTOR    lq_emp
21315  Advanced Manufacturing  0.498329


In [7]:
test2 = panel.copy()

# replicate what compute_location_quotient does step by step
total = (
    test2[test2["IS8_SECTOR"] == "Total"]
    [["YEAR", "GEOGRAPHY_CODE", "EMPLOYEES"]]
    .rename(columns={"EMPLOYEES": "TOTAL_EMP"})
)

national = (
    test2[test2["IS8_SECTOR"] != "Total"]
    .groupby(["YEAR", "IS8_SECTOR"], as_index=False)["EMPLOYEES"]
    .sum()
    .rename(columns={"EMPLOYEES": "NAT_IS8_EMP"})
)

nat_total = (
    test2[test2["IS8_SECTOR"] == "Total"]
    .groupby("YEAR", as_index=False)["EMPLOYEES"]
    .sum()
    .rename(columns={"EMPLOYEES": "NAT_TOTAL_EMP"})
)

print(f"Total rows: {total.shape}")
print(f"National IS8 rows: {national.shape}")
print(f"National total rows: {nat_total.shape}")
print(f"\nSample total:\n{total.head(3)}")
print(f"\nSample national:\n{national.head(3)}")
print(f"\nSample nat_total:\n{nat_total.head(3)}")

Total rows: (3850, 3)
National IS8 rows: (77, 3)
National total rows: (11, 2)

Sample total:
    YEAR GEOGRAPHY_CODE  TOTAL_EMP
7   2015      E06000001      29850
15  2015      E06000002      59920
23  2015      E06000003      41600

Sample national:
   YEAR              IS8_SECTOR  NAT_IS8_EMP
0  2015  Advanced Manufacturing       853450
1  2015     Creative Industries      1359635
2  2015                 Defence        17685

Sample nat_total:
   YEAR  NAT_TOTAL_EMP
0  2015       28716755
1  2016       29205410
2  2017       29517910


In [8]:
# continue from previous cell
test3 = test2[test2["IS8_SECTOR"] != "Total"].copy()
print(f"Non-total rows before merge: {test3.shape}")

test3 = test3.merge(total, on=["YEAR", "GEOGRAPHY_CODE"], how="left")
print(f"After merging total: {test3['TOTAL_EMP'].notna().sum()} non-null TOTAL_EMP")

test3 = test3.merge(national, on=["YEAR", "IS8_SECTOR"], how="left")
print(f"After merging national: {test3['NAT_IS8_EMP'].notna().sum()} non-null NAT_IS8_EMP")

test3 = test3.merge(nat_total, on="YEAR", how="left")
print(f"After merging nat_total: {test3['NAT_TOTAL_EMP'].notna().sum()} non-null NAT_TOTAL_EMP")

test3["lq_emp"] = (
    (test3["EMPLOYEES"] / test3["TOTAL_EMP"]) /
    (test3["NAT_IS8_EMP"] / test3["NAT_TOTAL_EMP"])
)
print(f"\nNon-null lq_emp: {test3['lq_emp'].notna().sum()}")
print(f"\nSample:")
print(test3[
    (test3['GEOGRAPHY_NAME'] == 'Sheffield') &
    (test3['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (test3['YEAR'] == 2023)
][['lq_emp', 'EMPLOYEES', 'TOTAL_EMP', 'NAT_IS8_EMP', 'NAT_TOTAL_EMP']])

Non-total rows before merge: (26950, 53)
After merging total: 24500 non-null TOTAL_EMP
After merging national: 26950 non-null NAT_IS8_EMP
After merging nat_total: 26950 non-null NAT_TOTAL_EMP

Non-null lq_emp: 24500

Sample:
         lq_emp  EMPLOYEES  TOTAL_EMP  NAT_IS8_EMP  NAT_TOTAL_EMP
21315  0.498329       3575     269950       832135       31312460


In [9]:
test4 = panel.copy()
print(f"Before emp_share: {test4.shape}, Total rows: {(test4['IS8_SECTOR']=='Total').sum()}")
test4 = builder.compute_employment_share(test4)
print(f"After emp_share: {test4.shape}, Total rows: {(test4['IS8_SECTOR']=='Total').sum()}")

Before emp_share: (30800, 53), Total rows: 3850
After emp_share: (30800, 54), Total rows: 3850


In [10]:
# Verify growth_emp for Sheffield Advanced Manufacturing
sheffield_emp = panel_ind[
    (panel_ind['GEOGRAPHY_NAME'] == 'Sheffield') &
    (panel_ind['IS8_SECTOR'] == 'Advanced Manufacturing')
][['YEAR', 'EMPLOYEES', 'growth_emp', 'cagr_emp']].dropna(subset=['growth_emp'])

print(sheffield_emp[['YEAR', 'EMPLOYEES', 'growth_emp', 'cagr_emp']].drop_duplicates(subset=['growth_emp']))

# Manual check: (2024 value - 2015 value) / 2015 value
emp_2015 = panel[
    (panel['GEOGRAPHY_NAME'] == 'Sheffield') &
    (panel['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (panel['YEAR'] == 2015)
]['EMPLOYEES'].values[0]

emp_2024 = panel[
    (panel['GEOGRAPHY_NAME'] == 'Sheffield') &
    (panel['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (panel['YEAR'] == 2024)
]['EMPLOYEES'].values[0]

manual_growth = (emp_2024 - emp_2015) / emp_2015
print(f"\nManual growth_emp: {manual_growth:.6f}")

      YEAR  EMPLOYEES  growth_emp  cagr_emp
1715  2015       3540   -0.011299 -0.001262

Manual growth_emp: -0.011299


In [11]:
print([c for c in panel.columns if "work" in c.lower() or "pop" in c.lower() or "labour" in c.lower() or "active" in c.lower()])

['active_enterprises']


In [12]:
print([c for c in panel.columns if c not in ["YEAR", "GEOGRAPHY_CODE", "GEOGRAPHY_NAME", "IS8_SECTOR", "EMPLOYEES", "BUSINESSES", "lq_emp", "lq_bus", "emp_share", "growth_emp", "cagr_emp", "growth_bus", "cagr_bus", "business_density", "related_variety", "size_large_share", "size_micro_share"]])

['gva_per_hour', 'weekly_pay', 'employment_rate', 'unemployment_rate', 'gdhi_per_head', 'new_enterprises', 'deaths_of_enterprises', 'active_enterprises', 'high_growth_enterprises', 'public_transport_to_employer', 'drive_to_employer', 'cycle_to_employer', 'broadband_availability', '4g_area_coverage', 'ks2_attainment', 'gcse_by_age_19', 'ofsted', 'persistent_absences', 'persistent_absences_fsm', 'persistent_absences_cla', 'early_years_comms', 'early_years_literacy', 'early_years_maths', 'fe_and_skills_achievements', 'apprenticeship_starts', 'apprenticeship_achievements', 'level_3+_qualifications', 'fe_and_skills_participation', 'female_hle', 'male_hle', 'smokers', 'reception_obesity', 'year_6_obesity', 'adult_obesity', 'cancer_diagnosis', 'under_75_mortality_rate', 'life_satisfaction', 'worthwhile', 'happiness', 'anxiety', 'net_additions']


In [13]:
import pandas as pd
from pathlib import Path

bus_raw = pd.read_parquet(root / "data/raw_data/business_counts/Business_counts_IS8_LADs.parquet")
print(bus_raw["SIZE_BAND"].unique())
print(bus_raw["SIZE_BAND"].value_counts())

<ArrowStringArray>
['large', 'medium', 'micro', 'small']
Length: 4, dtype: str
SIZE_BAND
large     312400
medium    312400
micro     312400
small     312400
Name: count, dtype: int64
